In [ ]:
import matplotlib.pyplot as plt 
import numpy as np

nb_agents = 1000
nb_secteurs = 10

A_brut = np.random.uniform(0, 1, (nb_agents, nb_secteurs))
A = A_brut / A_brut.sum(axis=1, keepdims=True)

In [ ]:
def Y(P, Pi):
    return P * Pi 

def J(Y_mat): 
    return np.argmax(Y_mat, axis=1)

def B(P, Pi):
    return np.max(Y(P, Pi), axis=1)

def Offre(P, Pi):
    Omega = np.zeros(nb_secteurs)
    Y_mat = Y(P, Pi)
    choix = J(Y_mat)
    for j in range(nb_secteurs):
        Omega[j] = np.sum(Y_mat[choix == j, j])
    return Omega

def Demande(A, B_vec):
    return B_vec @ A

def esperances(A, Pi, n=10000):
    acc_Offre = np.zeros(nb_secteurs)
    acc_Demande = np.zeros(nb_secteurs)
    
    for i in range(n):
        P_iter = np.random.uniform(0, 1, (nb_agents, nb_secteurs)) #######Attention, P n'est défini qu'ici 
        acc_Offre += Offre(P_iter, Pi)
        acc_Demande += Demande(A, B(P_iter, Pi))
        
    return acc_Offre / n, acc_Demande / n

In [ ]:
def algo_naif(A, n_iter=2000, learning_rate=0.01):
    Pi = np.ones(nb_secteurs) / nb_secteurs
    historique_Pi = [Pi.copy()]
    
    for i in range(n_iter):
        E_Offre, E_Demande = esperances(A, Pi)
        Z = E_Demande - E_Offre 
        
        Pi = Pi + learning_rate * Z
        
        Pi = np.maximum(Pi, 0.0001) 
        Pi = Pi / np.sum(Pi) 
        
        historique_Pi.append(Pi.copy())
        
        erreur = np.max(np.abs(Z))
        if erreur < 0.05: 
            print(f"Équilibre atteint à l'itération {i}")
            break
            
    return Pi, np.array(historique_Pi)

Pi_eq, historique = algo_naif(A)